In [ ]:
# clean_old_VM.ipynb
# Kill THIS machine's training processes and revoke their VM leases so a
# restart can begin IMMEDIATELY -- no 10-minute lease wait, no _2 name suffix,
# no peers sliding to lower-priority combos while zombie claims look live.
#
# When to run:
#   * BEFORE a planned restart: kills supervisors/workers here, revokes leases.
#   * AFTER a pod reboot: processes are already gone; it just revokes this
#     host's leftover leases. (Host matching needs registry records with
#     info.host -- written by supervisors from commit ecba730 on. For older
#     records it falls back to live-pid matching, or set VM_NAMES manually.)
#
# Old-code supervisors without the SIGTERM self-revoke are exactly what this
# notebook covers. Safe for the rest of the fleet: it only touches leases it
# can positively attribute to THIS machine (or names you list explicitly).
import os

APPLY = False        # dry-run first: run all cells, review the plan, then set
                     # True and re-run all cells to actually kill + revoke.
VM_NAMES = None      # e.g. ['VM5', 'VM5_2'] to force names; None = auto-detect

REPO = "/workspace/stable-query-latent"
OUT_DIR = "VICReg_review/heads/cloud_full_sweep_a100"
VM_DIR = os.path.join(REPO, OUT_DIR, "VM_parallel")

print('apply   :', APPLY)
print('vm names:', VM_NAMES or 'auto (info.host match, else live-pid match)')
print('vm dir  :', VM_DIR)


In [ ]:
# Scan: local training processes + which registry leases belong to this host.
import glob, json, socket, time
import psutil

HOST = socket.gethostname()
PATTERNS = ('sweep/supervisor.py', 'sweep/worker.py')


def local_train_procs():
    procs = []
    for p in psutil.process_iter(['pid', 'cmdline']):
        try:
            cmd = ' '.join(p.info['cmdline'] or [])
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            continue
        if any(pat in cmd for pat in PATTERNS):
            procs.append((p.info['pid'], 'worker' if 'worker.py' in cmd else 'supervisor', cmd[:150]))
    return procs


procs = local_train_procs()
local_pids = {pid for pid, _k, _c in procs}

targets = {}   # vm name -> registry payload
for f in sorted(glob.glob(os.path.join(VM_DIR, '*.json'))):
    try:
        rec = json.load(open(f, encoding='utf-8')) or {}
    except Exception:
        continue
    vm = rec.get('vm') or os.path.basename(f)[:-5]
    if VM_NAMES is not None:
        if vm in VM_NAMES:
            targets[vm] = rec
        continue
    host = (rec.get('info') or {}).get('host')
    if host == HOST:
        targets[vm] = rec                                   # new records: host match
    elif host is None and rec.get('pid') in local_pids:
        targets[vm] = rec                                   # legacy: live pid on this box

now = time.time()
print(f'host {HOST}: {len(procs)} local training process(es)')
for pid, kind, cmd in procs:
    print(f'  [{kind:10}] pid {pid}: {cmd}')
print(f'{len(targets)} lease(s) attributed to this machine:')
for vm, rec in targets.items():
    exp = float(rec.get('expiry', 0) or 0)
    state = f'fresh {exp - now:+.0f}s' if exp > now else 'already expired'
    print(f'  {vm:16} pid={rec.get("pid")} lease {state}')
if not targets and VM_NAMES is None and not procs:
    print('  (nothing found -- for pre-ecba730 records after a reboot, set VM_NAMES manually)')
if not APPLY:
    print()
    print('DRY-RUN: nothing killed or revoked. Set APPLY=True in the config cell')
    print('and re-run all cells to execute this plan.')


In [ ]:
# Apply: SIGTERM (new-code supervisors self-revoke) -> SIGKILL survivors
# (workers by process group, so their data-pool children die too) -> revoke
# leases -> verify. After this cell reports clean, restart training NOW.
if not APPLY:
    print('APPLY=False -- skipped.')
else:
    import signal

    for pid, kind, _cmd in procs:
        try:
            os.kill(pid, signal.SIGTERM)
        except OSError:
            pass
    time.sleep(5)
    for pid, kind, _cmd in procs:
        try:
            if kind == 'worker':
                os.killpg(os.getpgid(pid), signal.SIGKILL)   # + its data-pool children
            else:
                os.kill(pid, signal.SIGKILL)
            print(f'SIGKILLed {kind} {pid}')
        except (OSError, ProcessLookupError):
            pass                                             # exited on SIGTERM

    revoked_names, expired_names = [], []
    for vm in targets:
        f = os.path.join(VM_DIR, f'{vm}.json')
        try:
            rec = json.load(open(f, encoding='utf-8')) or {}
        except Exception:
            continue
        if float(rec.get('expiry', 0) or 0) <= time.time():
            expired_names.append(vm)
            print(f'{vm}: lease already expired')
            continue
        rec['expiry'] = 0.0
        rec['revoked_at'] = time.strftime('%Y-%m-%dT%H:%M:%S')
        rec['revoked_by'] = f'clean_old_VM@{HOST}'
        tmp = f + f'.tmp.{HOST}.{os.getpid()}'
        with open(tmp, 'w', encoding='utf-8') as h:
            json.dump(rec, h, ensure_ascii=False, indent=2)
        os.replace(tmp, f)
        revoked_names.append(vm)
        print(f'{vm}: lease revoked')

    leftovers = local_train_procs()
    print()
    print(f'summary: killed {len(procs) - len(leftovers)}/{len(procs)} process(es); '
          f'revoked {len(revoked_names)} lease(s) {revoked_names or "-"}; '
          f'already expired {expired_names or "-"}')
    if leftovers:
        for pid, kind, cmd in leftovers:
            print(f'!! still alive: [{kind}] {pid} {cmd}')
        print('NOT clean: kill the survivors manually before restarting.')
    elif not revoked_names and not expired_names:
        print('!! processes are gone, but NO lease was revoked (attribution found none).')
        print('   If this machine held a fresh lease it will linger up to 10 minutes;')
        print('   to revoke it explicitly, set VM_NAMES=[<its name>] and re-run all cells.')
    else:
        print('clean: processes gone AND leases handled. Safe to restart training')
        print('IMMEDIATELY -- no lease wait, same VM name reused, claims resume.')
